# NeoOLAF XQuality Machine 32 layer-wise ablation notebook

This notebook is intended to live at:

```text
NeoOLAF/examples/XQualityMachine32/layer_ablation_machine32.ipynb
```

It assumes `NeoOLAF` is the repository root and that the package source is under `src/`.

Main uses:

1. Run full or partial NeoOLAF pipelines on the Machine 32 PDF.
2. Resume from a saved layer `state.json`.
3. Review prompt size and prompt content layer by layer.
4. Evaluate each saved layer state against XQuality Machine 32 ground truth in JSON or Excel format.


In [15]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Document profile

This notebook uses the XQuality document-profile family. The default `PROFILE` is `xquality_loose`, with alternatives `xquality_strict_extraction` and `xquality_relaxed_recall`. These profiles keep XQuality-specific table, ontology-linking, and evaluation assumptions outside NeoOLAF core layers.


In [16]:
from pathlib import Path
import sys
import json
import pandas as pd

# Notebook location: NeoOLAF/examples/XQualityMachine32/
ROOT = Path.cwd()
if ROOT.name == "XQualityMachine32":
    ROOT = ROOT.parents[1]
elif (ROOT / "src" / "neoolaf").exists():
    ROOT = ROOT
else:
    # Useful when running from another working directory.
    ROOT = Path("../..").resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("NeoOLAF root:", ROOT)
print("src exists:", SRC.exists())

NeoOLAF root: c:\Users\henri\Documents\git\post-doc\NeoOLAF
src exists: True


## 1. Configure paths

Update these paths to match your local repository. The notebook accepts either a JSON ground-truth file or an Excel file. If an Excel file is used, it is converted automatically to JSON next to the original file.


In [17]:
# Input PDF and gold truth
PDF_PATH = ROOT / "data" / "XQuality" / "Textual" / "Chapitre_8_Alarmes_et_messages.pdf"
GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
GOLD_EXCEL = ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xlsx"

# Seed ontology, optional for the current minimal CLI.
SEED_ONTOLOGY = ROOT / "data" / "ontology" / "ContextOntology-COInd4.owl"

# Run folders
RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "full"
EVAL_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "eval_layers"

# Model and profile
MODEL = "openrouter/openai/gpt-oss-20b"
PROFILE = "xquality_loose"  # alternatives: xquality_strict_extraction, xquality_loose, xquality_relaxed_recall
DOCUMENT_PROFILE = PROFILE  # the CLI profile used by NeoOLAF

# Layer 1 execution toggles. Both are enabled by default for fast exploratory runs.
ENABLE_LAYER01_PARALLELISM = True
MAX_CONCURRENCY_LAYER01 = 4
RETRY_FAILED_CALLS = 3
RETRY_SLEEP_SECONDS = 2

ENABLE_LAYER01_RAG = True
RAG_BACKEND = "agentic" if ENABLE_LAYER01_RAG else "none"
RAG_TOP_K = 1
RAG_MAX_CHARS = 120

# Helper strings injected into the !python commands below.
LAYER01_PARALLEL_FLAGS = (
    f"--max-concurrency-layer01 {MAX_CONCURRENCY_LAYER01} "
    f"--retry-failed-calls {RETRY_FAILED_CALLS} "
    f"--retry-sleep-seconds {RETRY_SLEEP_SECONDS}"
    if ENABLE_LAYER01_PARALLELISM
    else ""
)
LAYER01_RAG_FLAGS = (
    f"--rag-layer01 --rag-top-k {RAG_TOP_K} --rag-max-chars {RAG_MAX_CHARS}"
    if ENABLE_LAYER01_RAG
    else ""
)

# Optional structured output / Pydantic validation for Layer 1.
# Profiles are the default source of truth; these flags make ablations easier.
ENABLE_LAYER01_STRUCTURED_OUTPUT = True
USE_LITELLM_RESPONSE_FORMAT_LAYER01 = False  # safer default with OpenRouter/local models
STRICT_STRUCTURED_OUTPUT_LAYER01 = False

LAYER01_STRUCTURED_OUTPUT_FLAGS = (
    "--structured-output-layer01 on"
    if ENABLE_LAYER01_STRUCTURED_OUTPUT
    else "--structured-output-layer01 off"
)
if USE_LITELLM_RESPONSE_FORMAT_LAYER01:
    LAYER01_STRUCTURED_OUTPUT_FLAGS += " --litellm-response-format-layer01"
if STRICT_STRUCTURED_OUTPUT_LAYER01:
    LAYER01_STRUCTURED_OUTPUT_FLAGS += " --strict-structured-output-layer01"


# Optional rerun-only-failed-chunks support.
# After a failed Layer 1 run, this manifest is saved automatically.
LAYER01_FAILED_CHUNKS_FILE = (
    ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer01_only"
    / "layer01_linguistic_expression_extraction" / "failed_chunks.json"
)
ENABLE_RERUN_ONLY_FAILED_CHUNKS = False
LAYER01_FAILED_CHUNKS_FLAGS = (
    f"--layer01-failed-chunks-file {LAYER01_FAILED_CHUNKS_FILE}"
    if ENABLE_RERUN_ONLY_FAILED_CHUNKS and LAYER01_FAILED_CHUNKS_FILE.exists()
    else ""
)

# Layer 2 and Layer 3 execution toggles.
# For XQuality, Layer 2 is now conservative ontology-aware normalization,
# and Layer 3 is role-based ontology-aware typing. Parallelism remains useful
# for generic profiles, and harmless here when the strategy is deterministic.
ENABLE_LAYER02_PARALLELISM = True
ENABLE_LAYER03_PARALLELISM = True
MAX_CONCURRENCY_LAYER02 = 4
MAX_CONCURRENCY_LAYER03 = 4

LAYER02_03_PARALLEL_FLAGS = (
    f"--max-concurrency-layer02 {MAX_CONCURRENCY_LAYER02} "
    f"--max-concurrency-layer03 {MAX_CONCURRENCY_LAYER03} "
    f"--retry-failed-calls {RETRY_FAILED_CALLS} "
    f"--retry-sleep-seconds {RETRY_SLEEP_SECONDS}"
)

# Optional rerun-only-failed support for Layer 2 and Layer 3.
LAYER02_FAILED_EXPRESSIONS_FILE = (
    ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer02_03_from_l1"
    / "layer02_candidate_enrichment" / "failed_expressions.json"
)
LAYER03_FAILED_ITEMS_FILE = (
    ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer02_03_from_l1"
    / "layer03_candidate_typing_resolution" / "failed_items.json"
)
ENABLE_RERUN_ONLY_FAILED_LAYER02 = False
ENABLE_RERUN_ONLY_FAILED_LAYER03 = False
LAYER02_FAILED_FLAGS = (
    f"--layer02-failed-expressions-file {LAYER02_FAILED_EXPRESSIONS_FILE}"
    if ENABLE_RERUN_ONLY_FAILED_LAYER02 and LAYER02_FAILED_EXPRESSIONS_FILE.exists()
    else ""
)
LAYER03_FAILED_FLAGS = (
    f"--layer03-failed-items-file {LAYER03_FAILED_ITEMS_FILE}"
    if ENABLE_RERUN_ONLY_FAILED_LAYER03 and LAYER03_FAILED_ITEMS_FILE.exists()
    else ""
)

print("PDF_PATH:", PDF_PATH)
print("GOLD_JSON:", GOLD_JSON)
print("GOLD_EXCEL:", GOLD_EXCEL)
print("RUN_DIR:", RUN_DIR)
print("DOCUMENT_PROFILE:", DOCUMENT_PROFILE)
print("Evaluation PROFILE:", PROFILE)
print("Parallel Layer 1:", ENABLE_LAYER01_PARALLELISM, "max_concurrency=", MAX_CONCURRENCY_LAYER01)
print("Layer 1 RAG:", ENABLE_LAYER01_RAG, "backend=", RAG_BACKEND, "top_k=", RAG_TOP_K, "max_chars=", RAG_MAX_CHARS)
print("Layer 1 structured output:", ENABLE_LAYER01_STRUCTURED_OUTPUT, "LiteLLM response_format=", USE_LITELLM_RESPONSE_FORMAT_LAYER01, "strict=", STRICT_STRUCTURED_OUTPUT_LAYER01)


print("Parallel Layer 2:", ENABLE_LAYER02_PARALLELISM, "max_concurrency=", MAX_CONCURRENCY_LAYER02)
print("Parallel Layer 3:", ENABLE_LAYER03_PARALLELISM, "max_concurrency=", MAX_CONCURRENCY_LAYER03)


PDF_PATH: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Textual\Chapitre_8_Alarmes_et_messages.pdf
GOLD_JSON: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json
GOLD_EXCEL: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.xlsx
RUN_DIR: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\full
DOCUMENT_PROFILE: xquality_loose
Evaluation PROFILE: xquality_loose
Parallel Layer 1: True max_concurrency= 4
Layer 1 RAG: True backend= agentic top_k= 1 max_chars= 120
Layer 1 structured output: True LiteLLM response_format= False strict= False
Parallel Layer 2: True max_concurrency= 4
Parallel Layer 3: True max_concurrency= 4


## 2. Optional: convert Excel ground truth to JSON

The evaluation helper accepts both `.json` and `.xlsx`, but converting once makes the experiment more reproducible.


In [18]:
from neoolaf.evaluation.runners.evaluate_layer_state import convert_xquality_excel_to_json

if GOLD_EXCEL.exists() and not GOLD_JSON.exists():
    convert_xquality_excel_to_json(GOLD_EXCEL, GOLD_JSON)
    print("Converted Excel ground truth to:", GOLD_JSON)
else:
    print("No conversion needed.")

No conversion needed.


## 3. Run the pipeline

The commands below use the CLI added for ablation. They save a restartable `state.json` after each executed layer.

Full run:


In [ ]:
# Full run, layers 0 to 12.
# Uncomment when ready to execute.

# !python -m neoolaf run \
#   --input-pdf "{PDF_PATH}" \
#   --run-dir "{RUN_DIR}" \
#   --from-layer 0 \
#   --to-layer 12 \
#   --model "{MODEL}" \
#   --rag-backend "{RAG_BACKEND}" \
#   --profile "{DOCUMENT_PROFILE}" \
#   {LAYER01_PARALLEL_FLAGS} \
#   {LAYER01_RAG_FLAGS} \
#   {LAYER01_STRUCTURED_OUTPUT_FLAGS} \
  {LAYER01_FAILED_CHUNKS_FLAGS} \
#   --verbose



### Partial examples

Layer 0 to 1 only:


In [ ]:
LAYER01_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer01_only"

!python -m neoolaf run \
  --input-pdf "{PDF_PATH}" \
  --run-dir "{LAYER01_RUN_DIR}" \
  --from-layer 0 \
  --to-layer 1 \
  --model "{MODEL}" \
  --rag-backend "{RAG_BACKEND}" \
  --profile "{DOCUMENT_PROFILE}" \
  {LAYER01_PARALLEL_FLAGS} \
  {LAYER01_RAG_FLAGS} \
  {LAYER01_STRUCTURED_OUTPUT_FLAGS} \
  {LAYER01_FAILED_CHUNKS_FLAGS} \
  --verbose



In [ ]:
# Optional: rerun only failed Layer 1 chunks from a previous run.
# This is useful when one provider/API failure drops a single record.
FAILED_RERUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer01_failed_only"
FAILED_CHUNKS_FILE = LAYER01_RUN_DIR / "layer01_linguistic_expression_extraction" / "failed_chunks.json"

if FAILED_CHUNKS_FILE.exists():
    print("Failed chunks manifest:", FAILED_CHUNKS_FILE)
    # Uncomment to rerun only failed chunks.
    # !python -m neoolaf run \
    #   --input-pdf "{PDF_PATH}" \
    #   --run-dir "{FAILED_RERUN_DIR}" \
    #   --from-layer 0 \
    #   --to-layer 1 \
    #   --model "{MODEL}" \
    #   --rag-backend "{RAG_BACKEND}" \
    #   --profile "{DOCUMENT_PROFILE}" \
    #   --layer01-failed-chunks-file "{FAILED_CHUNKS_FILE}" \
    #   {LAYER01_PARALLEL_FLAGS} \
    #   {LAYER01_RAG_FLAGS} \
    #   {LAYER01_STRUCTURED_OUTPUT_FLAGS} \
    #   --verbose
else:
    print("No failed chunks manifest found.")


Resume from layer 1 and run layers 2 and 3:


### Layer 2 and Layer 3, separated for debugging

Layer 2 enriches expressions with conservative normalization and ontology links. Layer 3 reads the Layer 2 state, performs role-based typing, and creates ontology-linked node and controlled relation candidates.


In [ ]:
LAYER02_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer02_from_l1"
LAYER01_STATE = LAYER01_RUN_DIR / "layer01_linguistic_expression_extraction" / "state.json"

!python -m neoolaf run \
   --resume-from "{LAYER01_STATE}" \
   --run-dir "{LAYER02_RUN_DIR}" \
   --from-layer 2 \
   --to-layer 2 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND}" \
   --profile "{DOCUMENT_PROFILE}" \
   {LAYER02_03_PARALLEL_FLAGS} \
   {LAYER02_FAILED_FLAGS} \
   --verbose


In [ ]:
LAYER03_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer03_from_l2"
LAYER02_STATE = LAYER02_RUN_DIR / "layer02_candidate_enrichment" / "state.json"

!python -m neoolaf run \
   --resume-from "{LAYER02_STATE}" \
   --run-dir "{LAYER03_RUN_DIR}" \
   --from-layer 3 \
   --to-layer 3 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND}" \
   --profile "{DOCUMENT_PROFILE}" \
   {LAYER02_03_PARALLEL_FLAGS} \
   {LAYER03_FAILED_FLAGS} \
   --verbose


In [ ]:
# Quick inspection of Layer 3 ontology-linked candidates.
LAYER03_OUTPUT = LAYER03_RUN_DIR / "layer03_candidate_typing_resolution" / "output.json"
if LAYER03_OUTPUT.exists():
    data = json.loads(LAYER03_OUTPUT.read_text(encoding="utf-8"))
    print("Entities:", len(data.get("entity_candidates", [])))
    print("Relations:", len(data.get("relation_candidates", [])))
    print("Events:", len(data.get("event_candidates", [])))
    for rel in data.get("relation_candidates", [])[:10]:
        print(rel.get("canonical_label"), rel.get("ontology_hints", [])[:4])
else:
    print("Layer 3 output not found yet:", LAYER03_OUTPUT)


In [ ]:
# Optional combined Layer 2+3 run. Keep separated cells above for debugging.
LAYER02_03_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer02_03_from_l1"
LAYER01_STATE = LAYER01_RUN_DIR / "layer01_linguistic_expression_extraction" / "state.json"

# !python -m neoolaf run \
#   --resume-from "{LAYER01_STATE}" \
#   --run-dir "{LAYER02_03_RUN_DIR}" \
#   --from-layer 2 \
#   --to-layer 3 \
#   --model "{MODEL}" \
#   --rag-backend "{RAG_BACKEND}" \
#   --profile "{DOCUMENT_PROFILE}" \
#   {LAYER02_03_PARALLEL_FLAGS} \
#   {LAYER02_FAILED_FLAGS} \
#   {LAYER03_FAILED_FLAGS} \
#   --verbose


In [5]:
LAYER06_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer06_from_l5"
LAYER07_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer07_from_l6"
LAYER06_STATE = LAYER06_RUN_DIR / "layer06_concept_relation_induction" / "state.json"

!python -m neoolaf run \
   --resume-from "{LAYER06_STATE}" \
   --run-dir "{LAYER07_RUN_DIR}" \
   --from-layer 7 \
   --to-layer 7 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND}" \
   --profile "{DOCUMENT_PROFILE}" \
   --max-concurrency-layer07 4 \
   --retry-failed-calls 3 \
   --retry-sleep-seconds 2 \
   --verbose

[NeoOLAF][Orchestrator] Starting execution plan
{
  "from_layer": 7,
  "to_layer": 7,
  "skip_layers": [],
  "mode": "pipeline",
  "max_concurrency_layer01": 1,
  "max_concurrency_layer02": 4,
  "max_concurrency_layer03": 4,
  "max_concurrency_layer04": 4,
  "max_concurrency_layer05": 4,
  "max_concurrency_layer06": 4,
  "max_concurrency_layer07": 4,
  "retry_failed_calls": 3,
  "retry_sleep_seconds": 2.0,
  "rag_backend": "agentic",
  "rag_layer01_enabled": false,
  "rag_top_k": 0,
  "rag_max_chars": 0,
  "metadata": {}
}
[NeoOLAF] Run directory: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer07_from_l6
[NeoOLAF] from_layer=7, to_layer=7, skip_layers=[]
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer07_hierarchisation']
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Pipeline finished in 0.39s
[

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [12]:
LAYER08_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer08_from_l7"
LAYER07_STATE = LAYER07_RUN_DIR / "layer07_hierarchisation" / "state.json"

!python -m neoolaf run \
   --resume-from "{LAYER07_STATE}" \
   --run-dir "{LAYER08_RUN_DIR}" \
   --from-layer 8 \
   --to-layer 8 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND}" \
   --profile "{DOCUMENT_PROFILE}" \
   --retry-failed-calls 3 \
   --retry-sleep-seconds 2 \
   --verbose

[NeoOLAF][Orchestrator] Starting execution plan
{
  "from_layer": 8,
  "to_layer": 8,
  "skip_layers": [],
  "mode": "pipeline",
  "max_concurrency_layer01": 1,
  "max_concurrency_layer02": 4,
  "max_concurrency_layer03": 4,
  "max_concurrency_layer04": 4,
  "max_concurrency_layer05": 4,
  "max_concurrency_layer06": 4,
  "max_concurrency_layer07": 4,
  "max_concurrency_layer08": 4,
  "retry_failed_calls": 3,
  "retry_sleep_seconds": 2.0,
  "rag_backend": "agentic",
  "rag_layer01_enabled": false,
  "rag_top_k": 0,
  "rag_max_chars": 0,
  "metadata": {}
}
[NeoOLAF] Run directory: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer08_from_l7
[NeoOLAF] from_layer=8, to_layer=8, skip_layers=[]
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer08_axiom_schemata_extraction']
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontolog

00:42:25 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
00:42:25 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [ ]:
LAYER09_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer09_from_l8"
LAYER08_STATE = LAYER08_RUN_DIR / "layer08_axiom_schemata_extraction" / "state.json"

!python -m neoolaf run \
   --resume-from "{LAYER08_STATE}" \
   --run-dir "{LAYER09_RUN_DIR}" \
   --from-layer 9 \
   --to-layer 9 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND}" \
   --profile "{DOCUMENT_PROFILE}" \
   --max-concurrency-layer09 4 \
   --retry-failed-calls 3 \
   --retry-sleep-seconds 2 \
   --verbose

^C


Run an ablation by skipping layers 8 and 9:


In [ ]:
ABLATION_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "skip_axiom_layers"
LAYER07_STATE = RUN_DIR / "layer07_hierarchisation" / "state.json"

# !python -m neoolaf run \
#   --resume-from "{LAYER07_STATE}" \
#   --run-dir "{ABLATION_RUN_DIR}" \
#   --from-layer 8 \
#   --to-layer 12 \
#   --skip-layers 8,9 \
#   --model "{MODEL}" \
#   --rag-backend "{RAG_BACKEND}" \
#   --profile "{DOCUMENT_PROFILE}" \
#   --verbose



## 3.1 Inspect Layer 1 record extraction

Quick sanity checks for record identifiers after Layer 1.


In [ ]:
def inspect_layer01_records(run_dir: Path) -> pd.DataFrame:
    output_path = run_dir / "layer01_linguistic_expression_extraction" / "output.json"
    if not output_path.exists():
        print("Layer 1 output not found:", output_path)
        return pd.DataFrame()
    data = json.loads(output_path.read_text(encoding="utf-8"))
    records = data.get("alarm_records", [])
    df = pd.DataFrame([
        {
            "record_id": r.get("record_id"),
            "record_type": r.get("record_type"),
            "alarm_no": r.get("alarm_no"),
            "message_no": r.get("message_no"),
            "label_en": r.get("alarm_label_en"),
            "chunk_id": r.get("chunk_id"),
            "page": r.get("page"),
        }
        for r in records
    ])
    print("records:", len(df))
    if not df.empty:
        print(df["record_type"].value_counts(dropna=False))
        display(df[df.duplicated(["record_type", "record_id"], keep=False)].sort_values(["record_type", "record_id"]))
    return df

layer01_df = inspect_layer01_records(LAYER01_RUN_DIR)


## 4. Inspect prompt size layer by layer

Each LLM layer now saves:

```text
<run_dir>/<layer_name>/prompt_stats.json
<run_dir>/<layer_name>/prompts/prompt_001_system.txt
<run_dir>/<layer_name>/prompts/prompt_001_user.txt
<run_dir>/<layer_name>/prompts/prompt_001_full.txt
```


In [ ]:
def collect_prompt_stats(run_dir: Path) -> pd.DataFrame:
    rows = []
    for stats_path in sorted(run_dir.glob("layer*/prompt_stats.json")):
        data = json.loads(stats_path.read_text(encoding="utf-8"))
        rows.append({
            "layer": stats_path.parent.name,
            "prompt_call_count": data.get("prompt_call_count", 0),
            "max_estimated_tokens": data.get("max_estimated_tokens", 0),
            "average_estimated_tokens": data.get("average_estimated_tokens", 0),
            "total_estimated_tokens": data.get("total_estimated_tokens", 0),
            "max_prompt_chars": data.get("max_prompt_chars", 0),
        })
    return pd.DataFrame(rows)

prompt_df = collect_prompt_stats(RUN_DIR)
prompt_df

In [ ]:
# Show the largest prompts first.
if not prompt_df.empty:
    display(prompt_df.sort_values("max_estimated_tokens", ascending=False))
else:
    print("No prompt_stats.json files found yet. Run the pipeline first.")

## 5. Evaluate every saved layer state against Machine 32 ground truth

The early layers are evaluated mostly as semantic label coverage. From layer 4 onward, relation-level metrics become more meaningful. From layer 5 onward, triple-level precision/recall/F1 is the main signal.


In [ ]:
from neoolaf.evaluation.runners.evaluate_layer_state import evaluate_run_layers

GOLD_PATH = GOLD_JSON if GOLD_JSON.exists() else GOLD_EXCEL
print("Using gold path:", GOLD_PATH)

results = evaluate_run_layers(
    run_dir=RUN_DIR,
    gold_path=GOLD_PATH,
    profile_name=PROFILE,
    output_dir=EVAL_DIR,
)

print(f"Evaluated {len(results)} layer states.")

In [ ]:
def summarize_layer_results(results: list[dict]) -> pd.DataFrame:
    rows = []
    for item in results:
        extraction = item.get("extraction", {})
        entity = extraction.get("entity", {})
        relation = extraction.get("relation", {})
        counts = extraction.get("counts", {})
        prompt_stats = item.get("prompt_stats", {})
        rows.append({
            "layer": item.get("layer_name"),
            "entity_P": entity.get("precision"),
            "entity_R": entity.get("recall"),
            "entity_F1": entity.get("f1"),
            "relation_P": relation.get("precision"),
            "relation_R": relation.get("recall"),
            "relation_F1": relation.get("f1"),
            "pred_entities": counts.get("pred_entities"),
            "gold_entities": counts.get("gold_entities"),
            "pred_relations": counts.get("pred_relations"),
            "gold_relations": counts.get("gold_relations"),
            "prompt_calls": prompt_stats.get("prompt_call_count", 0),
            "max_prompt_tokens": prompt_stats.get("max_estimated_tokens", 0),
        })
    return pd.DataFrame(rows)

summary_df = summarize_layer_results(results)
summary_df

In [ ]:
# Save a compact CSV summary for reporting.
EVAL_DIR.mkdir(parents=True, exist_ok=True)
summary_csv = EVAL_DIR / "layer_ablation_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

## 6. Plot layer-wise evolution

These plots help identify where recall, precision, or prompt size changes sharply.


In [ ]:
import matplotlib.pyplot as plt

if not summary_df.empty:
    ax = summary_df.plot(x="layer", y=["entity_F1", "relation_F1"], marker="o", figsize=(12, 5))
    ax.set_ylabel("F1")
    ax.set_title("Layer-wise entity and relation F1")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation summary available.")

In [ ]:
if not summary_df.empty:
    ax = summary_df.plot(x="layer", y="max_prompt_tokens", kind="bar", figsize=(12, 5), legend=False)
    ax.set_ylabel("Estimated tokens")
    ax.set_title("Maximum prompt size by layer")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No prompt statistics available.")

## 7. Inspect unmatched predictions for a specific layer

Use this to see what the layer is producing that does not match the ground truth.


In [ ]:
# Choose one layer result to inspect.
LAYER_TO_INSPECT = "layer05_candidate_triple_generation"

selected = next((item for item in results if item.get("layer_name") == LAYER_TO_INSPECT), None)
if selected is None:
    print("Layer result not found:", LAYER_TO_INSPECT)
else:
    unmatched = selected["extraction"].get("unmatched", {})
    print("Unmatched predicted relations, first 10:")
    for relation in unmatched.get("relations_pred", [])[:10]:
        print(relation)
    print("\nUnmatched gold relations, first 10:")
    for relation in unmatched.get("relations_gold", [])[:10]:
        print(relation)



## Layer 4: ontology-aware record relation extraction

In [ ]:
# Layer 4 only: ontology-aware record relation extraction from Layer 3 state
LAYER04_RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "layer04_from_l3"
LAYER03_STATE = LAYER03_RUN_DIR / "layer03_candidate_typing_resolution" / "state.json"

ENABLE_LAYER04_PARALLELISM = True
MAX_CONCURRENCY_LAYER04 = 4
ENABLE_LAYER04_RAG = True
RAG_BACKEND_LAYER04 = RAG_BACKEND if ENABLE_LAYER04_RAG else "none"

LAYER04_PARALLEL_FLAGS = (
    f"--max-concurrency-layer04 {MAX_CONCURRENCY_LAYER04}"
    if ENABLE_LAYER04_PARALLELISM
    else ""
)

!python -m neoolaf run \
   --resume-from "{LAYER03_STATE}" \
   --run-dir "{LAYER04_RUN_DIR}" \
   --from-layer 4 \
   --to-layer 4 \
   --model "{MODEL}" \
   --rag-backend "{RAG_BACKEND_LAYER04}" \
   --profile "{DOCUMENT_PROFILE}" \
   {LAYER04_PARALLEL_FLAGS} \
   --retry-failed-calls 3 \
   --retry-sleep-seconds 2 \
   --verbose
